# Análisis de Gentrificación - Barcelona

**Fecha**: 2026-01-06  
**Autor**: Barcelona Housing Demographics Analyzer  

Este notebook analiza los procesos de gentrificación en Barcelona, identificando barrios en riesgo y patrones de cambio urbano.

## Contenido
1. [Setup](#1-setup)
2. [Índice de Gentrificación](#2-indice)
3. [Análisis de Cambio de Precios](#3-precios)
4. [Presión Turística](#4-turismo)
5. [Desplazamiento Poblacional](#5-desplazamiento)
6. [Barrios en Riesgo](#6-riesgo)
7. [Mapas de Gentrificación](#7-mapas)
8. [Conclusiones](#8-conclusiones)

## 1. Setup <a id='1-setup'></a>

In [ ]:
# Imports
import sys
from pathlib import Path
import warnings
import json
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import shape
from scipy import stats

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

# Add project root to path
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.database import DatabaseManager

print("✅ Imports completados")

In [ ]:
# Conectar a la base de datos
db_manager = DatabaseManager()
conn = db_manager.get_connection()

print("✅ Conexión establecida")

## 2. Índice de Gentrificación <a id='2-indice'></a>

El **Índice de Gentrificación** combina múltiples indicadores:
- 📈 Cambio de precios (2012-2025)
- 🏨 Presión turística (Airbnb)
- 📉 Tasa de desempleo
- 👥 Cambio demográfico

**Escala**: 0-100 (mayor = más gentrificado)

In [ ]:
# Cargar datos consolidados para análisis de gentrificación
df_gent = pd.read_sql("""
    SELECT 
        b.barrio_id,
        b.barrio_nombre,
        b.distrito_nombre,
        
        -- Precios 2012 vs 2024
        AVG(CASE WHEN p.anio = 2012 THEN p.precio_m2_venta END) as precio_2012,
        AVG(CASE WHEN p.anio = 2024 THEN p.precio_m2_venta END) as precio_2024,
        
        -- Turismo
        SUM(CASE WHEN pt.anio = 2024 THEN pt.num_listings_airbnb ELSE 0 END) as airbnb_2024,
        SUM(CASE WHEN pt.anio = 2015 THEN pt.num_listings_airbnb ELSE 0 END) as airbnb_2015,
        
        -- Desempleo
        AVG(de.tasa_desempleo_estimada) as tasa_desempleo,
        
        -- Demografía
        AVG(d.poblacion_total) as poblacion
        
    FROM dim_barrios b
    LEFT JOIN fact_precios p ON b.barrio_id = p.barrio_id
    LEFT JOIN fact_presion_turistica pt ON b.barrio_id = pt.barrio_id
    LEFT JOIN fact_desempleo de ON b.barrio_id = de.barrio_id
    LEFT JOIN fact_demografia d ON b.barrio_id = d.barrio_id
    GROUP BY b.barrio_id, b.barrio_nombre, b.distrito_nombre
    HAVING precio_2012 IS NOT NULL AND precio_2024 IS NOT NULL
""", conn)

print(f"📊 Barrios analizados: {len(df_gent)}")
df_gent.head()

In [ ]:
# Calcular indicadores de gentrificación

# 1. Cambio de precios (% de incremento)
df_gent['cambio_precio_pct'] = ((df_gent['precio_2024'] - df_gent['precio_2012']) / df_gent['precio_2012'] * 100)

# 2. Cambio en turismo
df_gent['cambio_airbnb'] = df_gent['airbnb_2024'] - df_gent['airbnb_2015'].fillna(0)

# 3. Normalizar indicadores (0-100)
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 100))

# Indicadores positivos de gentrificación
df_gent['score_precio'] = scaler.fit_transform(df_gent[['cambio_precio_pct']])
df_gent['score_turismo'] = scaler.fit_transform(df_gent[['cambio_airbnb']])

# Indicador inverso (más desempleo = menos gentrificación)
df_gent['score_desempleo'] = 100 - scaler.fit_transform(df_gent[['tasa_desempleo']].fillna(df_gent['tasa_desempleo'].mean()))

# 4. Índice compuesto (ponderado)
df_gent['indice_gentrificacion'] = (
    df_gent['score_precio'] * 0.5 +        # 50% peso a cambio de precios
    df_gent['score_turismo'] * 0.3 +       # 30% peso a presión turística
    df_gent['score_desempleo'] * 0.2       # 20% peso a desempleo
).round(2)

# 5. Clasificación de riesgo
def clasificar_riesgo(score):
    if pd.isna(score):
        return 'Sin Datos'
    elif score >= 75:
        return 'Muy Alto'
    elif score >= 60:
        return 'Alto'
    elif score >= 40:
        return 'Medio'
    elif score >= 25:
        return 'Bajo'
    else:
        return 'Muy Bajo'

df_gent['riesgo_gentrificacion'] = df_gent['indice_gentrificacion'].apply(clasificar_riesgo)

print("✅ Índice de gentrificación calculado")
print(f"\n📊 Estadísticas del índice:")
print(df_gent['indice_gentrificacion'].describe())

In [ ]:
# Distribución del índice de gentrificación
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histograma
axes[0].hist(df_gent['indice_gentrificacion'], bins=20, color='coral', edgecolor='black', alpha=0.7)
axes[0].axvline(df_gent['indice_gentrificacion'].mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Media: {df_gent["indice_gentrificacion"].mean():.1f}')
axes[0].axvline(df_gent['indice_gentrificacion'].median(), color='green', linestyle='--', linewidth=2,
                label=f'Mediana: {df_gent["indice_gentrificacion"].median():.1f}')
axes[0].set_title('Distribución del Índice de Gentrificación', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Índice de Gentrificación (0-100)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Distribución por nivel de riesgo
riesgo_counts = df_gent['riesgo_gentrificacion'].value_counts()
colors_dict = {'Muy Alto': 'darkred', 'Alto': 'red', 'Medio': 'orange', 'Bajo': 'yellow', 'Muy Bajo': 'green', 'Sin Datos': 'lightgrey'}
riesgo_colors = [colors_dict.get(r, 'lightgrey') for r in riesgo_counts.index]

axes[1].bar(range(len(riesgo_counts)), riesgo_counts.values, color=riesgo_colors, edgecolor='black')
axes[1].set_xticks(range(len(riesgo_counts)))
axes[1].set_xticklabels(riesgo_counts.index, rotation=45)
axes[1].set_title('Barrios por Nivel de Riesgo', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Número de Barrios')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Distribución por nivel de riesgo:")
print(riesgo_counts)

## 3. Análisis de Cambio de Precios <a id='3-precios'></a>

In [ ]:
# Top 15 barrios con mayor incremento de precios
top_incremento = df_gent.nlargest(15, 'cambio_precio_pct')[['barrio_nombre', 'precio_2012', 'precio_2024', 'cambio_precio_pct']]

plt.figure(figsize=(12, 8))
plt.barh(range(len(top_incremento)), top_incremento['cambio_precio_pct'], color='darkred', edgecolor='black')
plt.yticks(range(len(top_incremento)), top_incremento['barrio_nombre'])
plt.xlabel('Incremento de Precio (%)', fontsize=12)
plt.title('Top 15 Barrios con Mayor Incremento de Precios (2012-2024)', fontsize=16, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n🏆 Top 10 barrios con mayor incremento:")
print(top_incremento.head(10).to_string())

In [ ]:
# Scatter plot: Precio inicial vs Incremento
plt.figure(figsize=(14, 8))
scatter = plt.scatter(df_gent['precio_2012'], df_gent['cambio_precio_pct'], 
                     c=df_gent['indice_gentrificacion'], cmap='RdYlGn_r', 
                     s=100, alpha=0.6, edgecolors='black')
plt.colorbar(scatter, label='Índice de Gentrificación')
plt.xlabel('Precio 2012 (€/m²)', fontsize=12)
plt.ylabel('Incremento de Precio (%)', fontsize=12)
plt.title('Relación entre Precio Inicial e Incremento (2012-2024)', fontsize=16, fontweight='bold')
plt.grid(alpha=0.3)

# Añadir línea de tendencia
z = np.polyfit(df_gent['precio_2012'], df_gent['cambio_precio_pct'], 1)
p = np.poly1d(z)
plt.plot(df_gent['precio_2012'].sort_values(), p(df_gent['precio_2012'].sort_values()), 
         "r--", alpha=0.8, linewidth=2, label='Tendencia')
plt.legend()

plt.tight_layout()
plt.show()

# Calcular correlación
corr = df_gent['precio_2012'].corr(df_gent['cambio_precio_pct'])
print(f"\n📊 Correlación precio inicial vs incremento: {corr:.3f}")

## 4. Presión Turística <a id='4-turismo'></a>

In [ ]:
# Análisis de presión turística
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10 barrios con más Airbnb
top_airbnb = df_gent.nlargest(10, 'airbnb_2024')[['barrio_nombre', 'airbnb_2024', 'cambio_airbnb']]
axes[0].barh(range(len(top_airbnb)), top_airbnb['airbnb_2024'], color='purple', edgecolor='black')
axes[0].set_yticks(range(len(top_airbnb)))
axes[0].set_yticklabels(top_airbnb['barrio_nombre'])
axes[0].set_xlabel('Listings Airbnb (2024)')
axes[0].set_title('Top 10 Barrios con Mayor Presión Turística', fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Correlación Airbnb vs Precio
axes[1].scatter(df_gent['airbnb_2024'], df_gent['precio_2024'], 
               c=df_gent['indice_gentrificacion'], cmap='RdYlGn_r',
               s=100, alpha=0.6, edgecolors='black')
axes[1].set_xlabel('Listings Airbnb (2024)')
axes[1].set_ylabel('Precio (€/m²) 2024')
axes[1].set_title('Correlación Turismo vs Precio', fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Correlación
corr_turismo = df_gent['airbnb_2024'].corr(df_gent['precio_2024'])
print(f"\n📊 Correlación Airbnb vs Precio: {corr_turismo:.3f}")

## 5. Desplazamiento Poblacional <a id='5-desplazamiento'></a>

In [ ]:
# Análisis de desempleo como proxy de desplazamiento
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Desempleo vs Gentrificación
axes[0].scatter(df_gent['tasa_desempleo'], df_gent['indice_gentrificacion'],
               c=df_gent['cambio_precio_pct'], cmap='RdYlGn_r',
               s=100, alpha=0.6, edgecolors='black')
axes[0].set_xlabel('Tasa de Desempleo (%)')
axes[0].set_ylabel('Índice de Gentrificación')
axes[0].set_title('Desempleo vs Gentrificación', fontweight='bold')
axes[0].grid(alpha=0.3)

# Boxplot por nivel de riesgo
riesgo_order = ['Muy Bajo', 'Bajo', 'Medio', 'Alto', 'Muy Alto']
df_gent_sorted = df_gent[df_gent['riesgo_gentrificacion'] != 'Sin Datos'].copy()
df_gent_sorted['riesgo_gentrificacion'] = pd.Categorical(df_gent_sorted['riesgo_gentrificacion'], 
                                                          categories=riesgo_order, ordered=True)
df_gent_sorted = df_gent_sorted.sort_values('riesgo_gentrificacion')

if len(df_gent_sorted) > 0:
    sns.boxplot(data=df_gent_sorted, x='riesgo_gentrificacion', y='tasa_desempleo', 
               palette=['green', 'yellow', 'orange', 'red', 'darkred'], ax=axes[1])
    axes[1].set_xlabel('Nivel de Riesgo')
    axes[1].set_ylabel('Tasa de Desempleo (%)')
    axes[1].set_title('Desempleo por Nivel de Riesgo', fontweight='bold')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Correlación
corr_desempleo = df_gent['tasa_desempleo'].corr(df_gent['indice_gentrificacion'])
print(f"\n📊 Correlación Desempleo vs Gentrificación: {corr_desempleo:.3f}")

## 6. Barrios en Riesgo <a id='6-riesgo'></a>

In [ ]:
# Identificar barrios en alto riesgo
alto_riesgo = df_gent[df_gent['riesgo_gentrificacion'].isin(['Alto', 'Muy Alto'])].sort_values('indice_gentrificacion', ascending=False)

print(f"🚨 BARRIOS EN ALTO RIESGO DE GENTRIFICACIÓN: {len(alto_riesgo)}\n")
print("=" * 100)

for idx, row in alto_riesgo.head(10).iterrows():
    print(f"\n{row['barrio_nombre']} ({row['distrito_nombre']})")
    print(f"  • Índice de Gentrificación: {row['indice_gentrificacion']:.1f}/100")
    print(f"  • Incremento de Precio: +{row['cambio_precio_pct']:.1f}%")
    print(f"  • Precio 2012: {row['precio_2012']:.0f}€/m² → 2024: {row['precio_2024']:.0f}€/m²")
    print(f"  • Listings Airbnb: {row['airbnb_2024']:.0f}")
    print(f"  • Tasa Desempleo: {row['tasa_desempleo']:.1f}%")
    print("-" * 100)

In [ ]:
# Visualización de barrios en riesgo
plt.figure(figsize=(14, 10))

# Crear scatter plot con tamaño proporcional a Airbnb
scatter = plt.scatter(df_gent['cambio_precio_pct'], df_gent['indice_gentrificacion'],
                     s=df_gent['airbnb_2024']*2,  # Tamaño proporcional a Airbnb
                     c=df_gent['tasa_desempleo'], cmap='RdYlGn_r',
                     alpha=0.6, edgecolors='black', linewidth=1.5)

plt.colorbar(scatter, label='Tasa de Desempleo (%)')
plt.xlabel('Incremento de Precio 2012-2024 (%)', fontsize=12)
plt.ylabel('Índice de Gentrificación (0-100)', fontsize=12)
plt.title('Mapa de Riesgo de Gentrificación\n(Tamaño = Presión Turística)', fontsize=16, fontweight='bold')

# Añadir líneas de referencia
plt.axhline(y=60, color='red', linestyle='--', alpha=0.5, label='Umbral Alto Riesgo')
plt.axhline(y=40, color='orange', linestyle='--', alpha=0.5, label='Umbral Medio Riesgo')

# Etiquetar barrios de muy alto riesgo
muy_alto_riesgo = df_gent[df_gent['riesgo_gentrificacion'] == 'Muy Alto']
for idx, row in muy_alto_riesgo.iterrows():
    plt.annotate(row['barrio_nombre'], 
                xy=(row['cambio_precio_pct'], row['indice_gentrificacion']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=8, alpha=0.8,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Mapas de Gentrificación <a id='7-mapas'></a>

In [ ]:
# Cargar geometrías
df_barrios_raw = pd.read_sql("""
    SELECT barrio_id, barrio_nombre, distrito_nombre, geometry_json
    FROM dim_barrios
    WHERE geometry_json IS NOT NULL
""", conn)

# Convertir a GeoDataFrame
geometries = []
valid_rows = []

for idx, row in df_barrios_raw.iterrows():
    try:
        if row['geometry_json']:
            geom_dict = json.loads(row['geometry_json'])
            geom = shape(geom_dict)
            geometries.append(geom)
            valid_rows.append(row.drop('geometry_json'))
    except:
        continue

if geometries:
    gdf_barrios = gpd.GeoDataFrame(valid_rows, geometry=geometries, crs='EPSG:4326')
    
    # Merge con datos de gentrificación
    gdf_gent = gdf_barrios.merge(df_gent[['barrio_id', 'indice_gentrificacion', 'riesgo_gentrificacion']], 
                                  on='barrio_id', how='left')
    
    print(f"✅ Geometrías cargadas: {len(gdf_gent)}")
else:
    print("⚠️  Sin geometrías disponibles")

In [ ]:
# Mapa de calor del índice de gentrificación
if 'gdf_gent' in locals() and len(gdf_gent) > 0:
    fig, ax = plt.subplots(1, 1, figsize=(16, 16))
    
    gdf_gent.plot(
        column='indice_gentrificacion',
        cmap='RdYlGn_r',
        legend=True,
        edgecolor='black',
        linewidth=0.8,
        ax=ax,
        legend_kwds={'label': 'Índice de Gentrificación (0-100)', 
                     'orientation': 'horizontal', 
                     'shrink': 0.8},
        missing_kwds={'color': 'lightgrey'}
    )
    
    ax.set_title('Mapa de Gentrificación de Barcelona', fontsize=20, fontweight='bold', pad=20)
    ax.axis('off')
    
    # Añadir etiquetas a barrios de muy alto riesgo
    muy_alto = gdf_gent[gdf_gent['riesgo_gentrificacion'] == 'Muy Alto']
    for idx, row in muy_alto.iterrows():
        if row.geometry and not pd.isna(row.geometry):
            centroid = row.geometry.centroid
            ax.annotate(row['barrio_nombre'],
                       xy=(centroid.x, centroid.y),
                       fontsize=9,
                       ha='center',
                       bbox=dict(boxstyle='round,pad=0.4', facecolor='red', alpha=0.7, edgecolor='darkred'),
                       color='white',
                       fontweight='bold')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Sin datos para visualizar")

In [ ]:
# Mapa por nivel de riesgo (categórico)
if 'gdf_gent' in locals() and len(gdf_gent) > 0:
    fig, ax = plt.subplots(1, 1, figsize=(16, 16))
    
    # Definir colores por nivel de riesgo
    color_map = {
        'Muy Alto': 'darkred',
        'Alto': 'red',
        'Medio': 'orange',
        'Bajo': 'yellow',
        'Muy Bajo': 'green',
        'Sin Datos': 'lightgrey'
    }
    
    # Rellenar NaN con 'Sin Datos' y mapear colores
    gdf_gent['riesgo_gentrificacion'] = gdf_gent['riesgo_gentrificacion'].fillna('Sin Datos')
    gdf_gent['color'] = gdf_gent['riesgo_gentrificacion'].map(color_map).fillna('lightgrey')
    
    gdf_gent.plot(
        color=gdf_gent['color'],
        edgecolor='black',
        linewidth=0.8,
        ax=ax
    )
    
    ax.set_title('Mapa de Riesgo de Gentrificación por Barrio', fontsize=20, fontweight='bold', pad=20)
    ax.axis('off')
    
    # Añadir leyenda manual
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=color, edgecolor='black', label=level) 
                      for level, color in color_map.items() if level != 'Sin Datos']
    ax.legend(handles=legend_elements, loc='upper left', title='Nivel de Riesgo', fontsize=12)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Sin datos para visualizar")

## 8. Conclusiones <a id='8-conclusiones'></a>

In [ ]:
# Generar resumen ejecutivo
print("=" * 100)
print("RESUMEN EJECUTIVO - ANÁLISIS DE GENTRIFICACIÓN EN BARCELONA")
print("=" * 100)

print(f"\n📊 ESTADÍSTICAS GENERALES")
print(f"  • Barrios analizados: {len(df_gent)}")
print(f"  • Índice medio de gentrificación: {df_gent['indice_gentrificacion'].mean():.1f}/100")
print(f"  • Incremento medio de precios: +{df_gent['cambio_precio_pct'].mean():.1f}%")

print(f"\n🚨 BARRIOS EN RIESGO")
for nivel in ['Muy Alto', 'Alto', 'Medio', 'Bajo', 'Muy Bajo']:
    count = len(df_gent[df_gent['riesgo_gentrificacion'] == nivel])
    pct = (count / len(df_gent)) * 100
    emoji = '🔴' if nivel in ['Muy Alto', 'Alto'] else '🟡' if nivel == 'Medio' else '🟢'
    print(f"  {emoji} {nivel}: {count} barrios ({pct:.1f}%)")

print(f"\n🏆 TOP 5 BARRIOS MÁS GENTRIFICADOS")
top_5 = df_gent.nlargest(5, 'indice_gentrificacion')
for i, (idx, row) in enumerate(top_5.iterrows(), 1):
    print(f"  {i}. {row['barrio_nombre']} - Índice: {row['indice_gentrificacion']:.1f}")

print(f"\n📈 FACTORES CLAVE")
print(f"  • Correlación Precio-Gentrificación: {df_gent['cambio_precio_pct'].corr(df_gent['indice_gentrificacion']):.3f}")
print(f"  • Correlación Turismo-Precio: {df_gent['airbnb_2024'].corr(df_gent['precio_2024']):.3f}")
print(f"  • Correlación Desempleo-Gentrificación: {df_gent['tasa_desempleo'].corr(df_gent['indice_gentrificacion']):.3f}")

print(f"\n💡 RECOMENDACIONES")
print(f"  1. Monitorear barrios con índice >60 (alto riesgo)")
print(f"  2. Implementar políticas de vivienda protegida en zonas críticas")
print(f"  3. Regular presión turística en barrios con >100 listings Airbnb")
print(f"  4. Apoyar a barrios con desempleo >8% para evitar desplazamiento")

print("\n" + "=" * 100)

In [ ]:
# Exportar resultados
output_dir = project_root / 'notebooks' / 'exports'
output_dir.mkdir(exist_ok=True, parents=True)

# Exportar CSV con resultados
df_gent_export = df_gent[[
    'barrio_nombre', 'distrito_nombre', 'indice_gentrificacion', 'riesgo_gentrificacion',
    'cambio_precio_pct', 'precio_2012', 'precio_2024', 
    'airbnb_2024', 'tasa_desempleo'
]].sort_values('indice_gentrificacion', ascending=False)

output_file = output_dir / 'analisis_gentrificacion_barcelona.csv'
df_gent_export.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"✅ Resultados exportados a: {output_file}")

In [ ]:
# Cerrar conexión
conn.close()
print("\n✅ Análisis de gentrificación completado")
print("✅ Conexión cerrada")